# Ex 1 : De Morgan's First Law equivalence

In [1]:
import pandas as pd
from sympy import symbols,pretty,Equivalent
from sympy.logic.boolalg import truth_table,BooleanFalse
p,q=symbols("p q")

In [2]:
def boolVal(bvars):
    if len(bvars)==1:
        return [[False],[True]]
    r=boolVal(bvars[1:])
    L=[]
    for l in r:
        L.append(l+[False])
        L.append(l+[True])
    return L

In [3]:
def Show_Truth_Table(Exprs,Vars):
    A=[]
    for row in truth_table(Vars[0],Vars):
        vL,vr=row
        A.append(vL)
    for exp in Exprs:
        for i,row in enumerate(truth_table(exp,Vars)):
            _,vr=row
            A[i].append(vr)
    return pd.DataFrame(A,columns=Vars+[pretty(exp) for exp in Exprs])

In [4]:
def are_equivalent(expr1, expr2, variables) -> bool:
    F=Equivalent(expr1,expr2)
    Bvals=boolVal(variables)
    for evaluation in Bvals:
        row_val=F.subs({var:val for var,val in zip(variables,evaluation)})
        if not row_val:
            return False
    return True

In [5]:
are_equivalent(~(p&q),(~p|~q),[p,q])

True

In [6]:
Show_Truth_Table([Equivalent(~(p&q),(~p|~q))],[p,q])

,p,q,¬(p ∧ q) ⇔ (¬p ∨ ¬q)
0,0,0,True
1,0,1,True
2,1,0,True
3,1,1,True


# Ex 2 De Morgan's Second Law equivalence

In [7]:
are_equivalent(~(p|q),~p&~q,[p,q])

True

In [8]:
Show_Truth_Table([Equivalent(~(p|q),~p&~q)],[p,q])

,p,q,(¬p ∧ ¬q) ⇔ ¬(p ∨ q)
0,0,0,True
1,0,1,True
2,1,0,True
3,1,1,True


De Morgan's Laws:
The negation of a disjunction is the conjuction of the negations, and the negation of a conjuction is the dijunction of the negations

# Ex 3 Double Negation simplify_logic

In [9]:
from sympy.logic.boolalg import simplify_logic

In [10]:
simplify_logic(~(~(p)))

p

# Ex 4 Absorption Laws simplify_logic

In [11]:
simplify_logic(p |(p & q))

p

In [12]:
simplify_logic(p&(p|q))

p

# Ex 5 Distributive Laws equivalence

In [13]:
r=symbols("r")

In [14]:
Ds1=Equivalent(p &(q | r) , (p & q) | (p & r))
Ds2=Equivalent(p | (q & r) ,(p | q) & (p | r))

In [15]:
Show_Truth_Table([Ds1,Ds2],[p,q,r])

,p,q,r,(p ∧ (q ∨ r)) ⇔ ((p ∧ q) ∨ (p ∧ r)),((p ∨ q) ∧ (p ∨ r)) ⇔ (p ∨ (q ∧ r))
0,0,0,0,True,True
1,0,0,1,True,True
2,0,1,0,True,True
3,0,1,1,True,True
4,1,0,0,True,True
5,1,0,1,True,True
6,1,1,0,True,True
7,1,1,1,True,True


# Ex 6 Simplification with simplify_logic

In [16]:
F61=(p & q) | (p & ~ q)
F62=(p | q)&(p | ~ q)

In [17]:
from sympy.logic import simplify_logic

In [18]:
simplify_logic(F61)

p

In [19]:
simplify_logic(F62)

p

# Conjunctive Normal Form (CNF) to_cnf


In [20]:
from sympy import to_cnf

In [22]:
F71=p>>q
F72=Equivalent(p,q)
F73=(p|q)&(p>>r)

In [27]:
CNF1=to_cnf(F71)
CNF1

q | ~p

In [28]:
CNF2=to_cnf(F72)
CNF2

(p | ~q) & (q | ~p)

In [29]:
CNF3=to_cnf(F73)
CNF3

(p | q) & (r | ~p)

In [32]:
print(are_equivalent(CNF1,F71,[p,q]))
print(are_equivalent(CNF2,F72,[p,q]))
print(are_equivalent(CNF3,F73,[p,q,r]))

True
True
True


# Ex 8 Disjunctive Normal Form (DNF) to_dnf

In [33]:
from sympy import to_dnf

In [35]:
F81=(p|q)&~r
F82=Equivalent(p,q&r)

In [37]:
DNF1=to_dnf(F81)
DNF1

(p & ~r) | (q & ~r)

In [39]:
DNF2=to_dnf(F82)
DNF2

(p & ~p) | (p & q & r) | (~p & ~q) | (~p & ~r) | (q & r & ~q) | (q & r & ~r)

In [42]:
print(are_equivalent(DNF1,F81,[p,q,r]))
print(are_equivalent(DNF2,F82,[p,q,r]))

True
True


In [43]:
simplify_logic(DNF2)

(p & q & r) | (~p & ~q) | (~p & ~r)

# Ex 9 Satisfiability satisfiable

In [95]:
from sympy.logic.inference import satisfiable

In [96]:
F9=[p&q,p&~p,(p>>q)&p&~q,p|~p]

In [97]:
for f in F9:
    if satisfiable(f):
        if not (satisfiable(~f)):
            print(f"{pretty(f)} is tautologie")
        else:
            print(f"{pretty(f)} is contingency")
        print("\ta modell for it ",satisfiable(f))
    else:
        print(f"{pretty(f)} is unsatisfiable")

p ∧ q is contingency
	a modell for it  {q: True, p: True}
p ∧ ¬p is unsatisfiable
p ∧ (p → q) ∧ ¬q is unsatisfiable
p ∨ ¬p is tautologie
	a modell for it  {p: False}


# Ex 10 Counting Function — At Least   of   combinations

In [177]:
from itertools import combinations
from sympy import And,Or

In [178]:
def at_least_k_of(variables, k):
    return Or(*[And(*combo) for combo in combinations(variables,k)] )

In [179]:
variables=symbols(" ".join(chr(ord("a")+i) for i in range(5)))

In [180]:
F=at_least_k_of(variables,2)

In [181]:
F

(a & b) | (a & c) | (a & d) | (a & e) | (b & c) | (b & d) | (b & e) | (c & d) | (c & e) | (d & e)

In [182]:
valuation1={symbols(chr(ord('a')+i)):True if i<2 else False for i in range(5)}
valuation2={symbols(chr(ord('a')+i)):True if i<1 else False for i in range(5)}

In [183]:
F.subs(valuation1)

True

In [184]:
F.subs(valuation2)

False

In [185]:
simplify_logic(F)

(a & b) | (a & c) | (a & d) | (a & e) | (b & c) | (b & d) | (b & e) | (c & d) | (c & e) | (d & e)

# Ex 11 Implication Equivalences equivalence

In [200]:
def message(F,Q,variables):
    if are_equivalent(F,Q,variables):
        print(pretty(F)," is equivalent to" ,pretty(Q))
    else:
        print(pretty(F)," is not equivalent to" ,pretty(Q))


In [201]:
message(p>>q,~p|q,[p,q])

p → q  is equivalent to q ∨ ¬p


In [202]:
message(p>>q,~q>>~p,[p,q])

p → q  is equivalent to ¬q → ¬p


In [204]:
message(~(p>>q),p&~q,[p,q])

p ↛ q  is equivalent to p ∧ ¬q


In [205]:
message(~(p>>q),p&q,[p,q])

p ↛ q  is not equivalent to p ∧ q


# Exercice 10 bis

In [53]:
def combinaisons(variables,k):
    if k==0 or len (variables)<k:
        return []
    if k==1:
        return [{v} for v in variables]
    A=combinaisons(variables[1:],k-1)
    B=combinaisons(variables[1:],k)
    L=[]
    L.extend(B)
    for a in A:
            L.append({variables[0]}.union(a))
    return L

In [54]:
combinaisons([1,2,3,4],3)

[{2, 3, 4}, {1, 3, 4}, {1, 2, 3}, {1, 2, 4}]